# Despacho de Recursos de Emergencia — Simulación (Camino A)

**Tarea N°3 · ICS2123 Modelos Estocásticos · PUC**

Modelo de simulación de eventos discretos (cola de pérdida, sin espera) para el despacho de
unidades de emergencia en la Región Metropolitana, a partir de datos reales (SIGEB).

---

## Índice de secciones

1. **Datos** — carga, limpieza y derivación de columnas de apoyo.
2. **Modelo** — parámetros empíricos (tasa horaria, unidades por emergencia, tiempo de servicio).
3. **Motor** — calendario de eventos con `heapq`; modos agregado y por comuna.
4. **Verificación** — tests determinísticos, saturación, conservación y validación de volumen/perfil.
5. **Experimentos** — 3 escenarios × 2 resoluciones, 30 réplicas, IC 95%.
6. **Sensibilidad** — barridos de tiempo de servicio, flota y demanda.
7. **Export** — CSV, figuras y artefactos de resultados.

In [1]:
# Imports base del proyecto
import heapq
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print('numpy', np.__version__)
print('pandas', pd.__version__)

numpy 2.5.0
pandas 3.0.3


## 1. Datos

Carga del dataset SIGEB, limpieza (filas con `Comuna`/`Material` nulos) y derivación de
columnas de apoyo (`hora`, `n_unidades`, `zona`). Salida: `datos_procesados.csv`.

In [2]:
# 2.1 Carga del dataset (ruta robusta con pathlib: funciona desde el notebook o la raíz)
def encontrar_xlsx(nombre='SIGEB_Estadisticas.xlsx'):
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base / 'base' / nombre, base / 'tarea-3' / 'base' / nombre):
            if cand.exists():
                return cand
    raise FileNotFoundError(f'No se encontró {nombre} en base/ desde {Path.cwd()}')

RUTA_XLSX = encontrar_xlsx()
DIR_SALIDA = RUTA_XLSX.parent.parent / 'despacho_recursos'
print('Dataset:', RUTA_XLSX)

df = pd.read_excel(RUTA_XLSX, sheet_name='Sheet1')
df['Fecha'] = pd.to_datetime(df['Fecha'])
print('Filas cargadas:', len(df))

Dataset: C:\Users\josea\Downloads\uc\2026-1\MMEE\UC-2026-1_MMEE-T3\tarea-3\base\SIGEB_Estadisticas.xlsx


Filas cargadas: 10082


In [3]:
# 2.2 Limpieza: quitar filas con Comuna o Material nulos; normalizar Comuna
n0 = len(df)
n_comuna_nula = int(df['Comuna'].isna().sum())
n_material_nulo = int(df['Material'].isna().sum())
mask_nulos = df['Comuna'].isna() | df['Material'].isna()
print(f'Comuna nula: {n_comuna_nula} | Material nulo: {n_material_nulo} | eliminadas (unión): {int(mask_nulos.sum())}')
df = df.loc[~mask_nulos].copy()
df['Comuna'] = df['Comuna'].str.strip().str.replace(r'\s+', ' ', regex=True)
print(f'Filas: {n0} -> {len(df)}')

Comuna nula: 3 | Material nulo: 1 | eliminadas (unión): 3
Filas: 10082 -> 10079


In [4]:
# 2.3 Columnas de apoyo: hora (0-23), n_unidades (conteo en Material), zona (9 desagregadas + Resto RM)
ZONAS_DESAGREGADAS = ['Santiago', 'Las Condes', 'Providencia', 'Estación Central', 'Renca',
                      'Independencia', 'Vitacura', 'Lo Barnechea', 'Recoleta']

df['hora'] = df['Fecha'].dt.hour
df['n_unidades'] = df['Material'].str.split(',').apply(lambda ts: sum(1 for t in ts if t.strip()))
df['zona'] = df['Comuna'].where(df['Comuna'].isin(ZONAS_DESAGREGADAS), 'Resto RM')

print('hora:', df['hora'].min(), '-', df['hora'].max())
print('n_unidades media:', round(df['n_unidades'].mean(), 4))
print('Distribución por zona:')
print(df['zona'].value_counts())

hora: 0 - 23
n_unidades media: 2.1451
Distribución por zona:
zona
Santiago            3412
Las Condes          1508
Providencia         1088
Estación Central    1049
Renca                730
Independencia        521
Vitacura             477
Lo Barnechea         472
Recoleta             443
Resto RM             379
Name: count, dtype: int64


In [5]:
# 2.4 Exportar dataset procesado (UTF-8)
RUTA_CSV = DIR_SALIDA / 'datos_procesados.csv'
df.to_csv(RUTA_CSV, index=False, encoding='utf-8')
print('Guardado:', RUTA_CSV)
print('Filas:', len(df), '| Columnas:', list(df.columns))

Guardado: C:\Users\josea\Downloads\uc\2026-1\MMEE\UC-2026-1_MMEE-T3\tarea-3\despacho_recursos\datos_procesados.csv
Filas: 10079 | Columnas: ['Correlativo', 'Fecha', 'Clave', 'Calle', 'Esquina', 'Comuna', 'Material', 'hora', 'n_unidades', 'zona']


## 2. Modelo — parámetros empíricos

Se estiman a partir del dataset procesado los tres insumos aleatorios del modelo:

- **`tasa_horaria`** (24): tasa media de llegadas por hora del día = (conteo en esa hora) / 365
  días. También su versión desagregada **`tasa_horaria_zona`** (10 zonas × 24 horas).
- **`unidades_muestra`**: vector empírico de unidades por emergencia (para bootstrap), y su
  versión por zona **`unidades_muestra_zona`**.

Todo se exporta a `parametros.npz` para que el motor los cargue sin recomputar.

In [6]:
# 3.1 Tasa horaria de llegada (emergencias/hora): conteo por hora del día / 365
DIAS_ANIO = 365
ZONAS_ORDEN = ZONAS_DESAGREGADAS + ['Resto RM']  # orden fijo para las matrices/arrays por zona

conteo_hora = df.groupby('hora').size().reindex(range(24), fill_value=0)
tasa_horaria = (conteo_hora / DIAS_ANIO).to_numpy()

# Matriz zona x hora (10 x 24) siguiendo ZONAS_ORDEN
piv = (df.groupby(['zona', 'hora']).size()
         .unstack('hora')
         .reindex(index=ZONAS_ORDEN, columns=range(24), fill_value=0))
tasa_horaria_zona = (piv / DIAS_ANIO).to_numpy()

print('tasa_horaria shape:', tasa_horaria.shape, '| suma (emergencias/dia):', round(tasa_horaria.sum(), 3))
print('tasa_horaria_zona shape:', tasa_horaria_zona.shape)
print('hora pico:', int(tasa_horaria.argmax()), '| hora valle:', int(tasa_horaria.argmin()))

tasa_horaria shape: (24,) | suma (emergencias/dia): 27.614
tasa_horaria_zona shape: (10, 24)
hora pico: 18 | hora valle: 4


In [7]:
# 3.2 Distribución empírica de unidades por emergencia (para bootstrap)
unidades_muestra = df['n_unidades'].to_numpy()
unidades_muestra_zona = [df.loc[df['zona'] == z, 'n_unidades'].to_numpy() for z in ZONAS_ORDEN]

print('unidades_muestra:', unidades_muestra.shape, '| media:', round(unidades_muestra.mean(), 4))
for z, arr in zip(ZONAS_ORDEN, unidades_muestra_zona):
    print(f'  {z:18s} n={len(arr):5d}  media={arr.mean():.3f}')

unidades_muestra: (10079,) | media: 2.1451
  Santiago           n= 3412  media=2.311
  Las Condes         n= 1508  media=2.005
  Providencia        n= 1088  media=2.124
  Estación Central   n= 1049  media=2.130
  Renca              n=  730  media=2.115
  Independencia      n=  521  media=2.332
  Vitacura           n=  477  media=2.046
  Lo Barnechea       n=  472  media=2.004
  Recoleta           n=  443  media=2.266
  Resto RM           n=  379  media=1.269


In [8]:
# 3.3 Exportar parámetros a parametros.npz
RUTA_NPZ = DIR_SALIDA / 'parametros.npz'
np.savez(
    RUTA_NPZ,
    tasa_horaria=tasa_horaria,
    tasa_horaria_zona=tasa_horaria_zona,
    unidades_muestra=unidades_muestra,
    unidades_muestra_zona=np.array(unidades_muestra_zona, dtype=object),
    zonas_orden=np.array(ZONAS_ORDEN),
)
print('Guardado:', RUTA_NPZ)

# Verificación de contenido
_chk = np.load(RUTA_NPZ, allow_pickle=True)
print('Claves:', list(_chk.files))

Guardado: C:\Users\josea\Downloads\uc\2026-1\MMEE\UC-2026-1_MMEE-T3\tarea-3\despacho_recursos\parametros.npz
Claves: ['tasa_horaria', 'tasa_horaria_zona', 'unidades_muestra', 'unidades_muestra_zona', 'zonas_orden']


## 3. Motor — simulación de eventos discretos (`heapq`)

El sistema se modela como una **cola de pérdida** (sin espera): cada emergencia que no
puede ser atendida de inmediato por falta de unidades se **bloquea** (no espera).

Bloque base (4.1–4.3): generación de llegadas y muestreos de las entradas aleatorias.
Estas funciones se definen aquí y se mantienen **idénticas** en `src/motor.py`.

- **Llegadas** — Poisson no homogéneo por hora del día vía *thinning*; las inter-llegadas
  candidatas se generan por **transformada inversa** de la exponencial.
- **Unidades por emergencia** — *bootstrap* de la distribución empírica.
- **Tiempo de ocupación por unidad** — Exponencial de media 45 min.

In [9]:
# 4.1-4.3 Generadores y muestreos base del motor (copia idéntica en src/motor.py)
def generar_llegadas(tasa_horaria, dias, rng):
    """Genera tiempos de llegada (en minutos) de un proceso de Poisson NO homogéneo
    por hora del día, usando el método de thinning (adelgazamiento).

    `tasa_horaria` es un array de largo 24 con la tasa media (emergencias/hora) de
    cada hora del día. Se toma una cota superior constante `lam_max` y se generan
    candidatos con esa tasa; cada candidato en el minuto t se acepta con probabilidad
    tasa(hora(t)) / lam_max, lo que reproduce la intensidad variable.

    Los tiempos entre candidatos se generan por TRANSFORMADA INVERSA de la
    exponencial: si U ~ Uniforme(0,1), entonces  t += -ln(1 - U) / lam_max  es una
    inter-llegada Exponencial(lam_max). (Requisito del enunciado: explicar cómo se
    genera una variable aleatoria relevante.)
    """
    lam_max = tasa_horaria.max() / 60.0          # cota superior (emergencias/minuto)
    T = dias * 24 * 60                            # horizonte en minutos
    llegadas = []
    t = 0.0
    while True:
        u = rng.random()
        t += -np.log(1.0 - u) / lam_max          # transformada inversa exponencial
        if t >= T:
            break
        hora = int((t // 60) % 24)
        if rng.random() < (tasa_horaria[hora] / 60.0) / lam_max:  # aceptación (thinning)
            llegadas.append(t)
    return llegadas


def muestrear_unidades(muestra, rng):
    """Número de unidades que requiere una emergencia, por bootstrap: se elige un
    valor al azar del vector empírico observado (distribución empírica)."""
    return int(muestra[rng.integers(len(muestra))])


def muestrear_servicio(media, rng):
    """Tiempo de ocupación de UNA unidad, Exponencial de media `media` (min)."""
    return float(rng.exponential(media))

In [10]:
# Verificación rápida del bloque (los tests formales van en la sección 4)
rng_demo = np.random.default_rng(42)
_ll = generar_llegadas(tasa_horaria, dias=7, rng=rng_demo)
print('Llegadas en 7 días:', len(_ll), '| esperado ~', round(7 * tasa_horaria.sum()))
print('Muestra unidades:', [muestrear_unidades(unidades_muestra, rng_demo) for _ in range(5)])
print('Muestra servicio (media 45):', [round(muestrear_servicio(45, rng_demo), 1) for _ in range(5)])

Llegadas en 7 días: 187 | esperado ~ 193
Muestra unidades: [1, 1, 1, 8, 1]
Muestra servicio (media 45): [6.6, 3.4, 51.1, 27.4, 70.9]


### 4.4 Motor de una réplica — modo AGREGADO

Una sola flota atiende toda la RM. El estado es el número de unidades `ocupadas`; cada
emergencia toma `need` unidades (muestreadas por bootstrap) si hay suficientes libres, y
agenda una liberación independiente `Exp(media_serv)` por unidad; si no, se bloquea. El
warm-up se procesa pero no se contabiliza. Métricas: `p_bloqueo` (%) y utilización media.

In [11]:
# 4.4 Motor de una réplica en modo AGREGADO (copia idéntica en src/motor.py)
def simular_replica_agregado(flota, tasa_horaria, unidades_muestra, media_serv,
                             dias, warmup, rng, debug=False):
    """Simula UNA réplica del sistema en modo AGREGADO (una sola flota para toda la RM).

    Cola de pérdida: si al llegar una emergencia no hay unidades suficientes libres,
    se bloquea (no espera). Motor de eventos discretos con `heapq`.

    Parámetros
    ----------
    flota : int              -- número total de unidades disponibles.
    tasa_horaria : array(24) -- tasa media de llegadas por hora del día.
    unidades_muestra : array -- vector empírico de unidades/emergencia (bootstrap).
    media_serv : float       -- media de la Exponencial del tiempo de ocupación (min).
    dias : int               -- días medidos (horizonte tras el warm-up).
    warmup : int             -- días iniciales que se procesan pero NO se cuentan.
    rng : Generator          -- numpy.random.default_rng(semilla).
    debug : bool             -- si True, chequea la invariante 0 <= ocupadas <= flota.

    Devuelve
    --------
    dict con p_bloqueo (%), utilizacion (0-1), atendidas, bloqueadas, n_contadas.
    """
    warmup_min = warmup * 24 * 60
    T_fin = (warmup + dias) * 24 * 60
    llegadas = generar_llegadas(tasa_horaria, warmup + dias, rng)

    # Calendario de eventos: (tiempo, orden, tipo, k). orden = desempate estable.
    # tipo 'A' = llegada (k=0), tipo 'L' = liberación de k unidades.
    heap = []
    orden = 0
    for t in llegadas:
        heapq.heappush(heap, (t, orden, 'A', 0)); orden += 1

    ocupadas = 0
    atendidas = 0
    bloqueadas = 0
    area = 0.0          # integral de ocupadas(t) dt en la ventana medida
    t_prev = 0.0

    while heap:
        t, _, tipo, k = heapq.heappop(heap)
        # Acumular utilización sobre [t_prev, t] recortado a la ventana [warmup_min, T_fin]
        lo = max(t_prev, warmup_min)
        hi = min(t, T_fin)
        if hi > lo:
            area += ocupadas * (hi - lo)
        t_prev = t

        if tipo == 'A':
            need = muestrear_unidades(unidades_muestra, rng)
            if ocupadas + need <= flota:
                ocupadas += need
                for _ in range(need):                       # una liberación por unidad
                    dur = muestrear_servicio(media_serv, rng)
                    heapq.heappush(heap, (t + dur, orden, 'L', 1)); orden += 1
                if t >= warmup_min:
                    atendidas += 1
            else:
                if t >= warmup_min:
                    bloqueadas += 1
        else:                                               # liberación
            ocupadas -= k

        if debug:
            assert 0 <= ocupadas <= flota, f'invariante rota: ocupadas={ocupadas}, flota={flota}'

    n_contadas = atendidas + bloqueadas
    p_bloqueo = 100.0 * bloqueadas / n_contadas if n_contadas else 0.0
    utilizacion = area / (flota * (T_fin - warmup_min)) if flota else 0.0
    return {'p_bloqueo': p_bloqueo, 'utilizacion': utilizacion,
            'atendidas': atendidas, 'bloqueadas': bloqueadas, 'n_contadas': n_contadas}

In [12]:
# Verificación rápida: monotonía (más flota => menos bloqueo)
for f in (10, 20, 30, 45):
    r = simular_replica_agregado(f, tasa_horaria, unidades_muestra, media_serv=45,
                                 dias=30, warmup=3, rng=np.random.default_rng(42), debug=True)
    print(f'flota={f:3d}  p_bloqueo={r["p_bloqueo"]:6.2f}%  utilizacion={r["utilizacion"]:.3f}  '
          f'(atendidas={r["atendidas"]}, bloqueadas={r["bloqueadas"]})')

flota= 10  p_bloqueo=  1.28%  utilizacion=0.149  (atendidas=770, bloqueadas=10)
flota= 20  p_bloqueo=  0.38%  utilizacion=0.079  (atendidas=777, bloqueadas=3)
flota= 30  p_bloqueo=  0.13%  utilizacion=0.055  (atendidas=779, bloqueadas=1)
flota= 45  p_bloqueo=  0.00%  utilizacion=0.037  (atendidas=780, bloqueadas=0)


### 4.5 Motor de una réplica — modo POR ZONA (con refuerzo)

Cada una de las 10 zonas tiene su propia flota y su propio contador de unidades ocupadas.
Si una zona no puede atender una emergencia con sus unidades, y `refuerzo=True`, usa todas
sus libres locales y pide el faltante a la **única zona con más unidades libres** (un solo
donante); si ese donante no alcanza, la emergencia se bloquea. Las unidades prestadas
vuelven a su zona dueña al liberarse. Métricas: `p_bloqueo` global y por zona, y `%` de
atenciones que requirieron refuerzo.

In [13]:
# 4.5 Motor de una réplica en modo POR ZONA (copia idéntica en src/motor.py)
def simular_replica_zonas(flota_por_zona, tasa_horaria_zona, unidades_muestra_zona,
                          media_serv, dias, warmup, rng, zonas, refuerzo=True, debug=False):
    """Simula UNA réplica en modo POR ZONA. Cada zona tiene su propia flota y su
    propio contador de unidades ocupadas.

    Al llegar una emergencia a la zona Z (índice zi):
      - si Z tiene suficientes unidades libres -> se atiende localmente;
      - si no y `refuerzo=True` -> se usan TODAS las libres locales y el faltante lo
        cubre la ÚNICA zona con más unidades libres (un solo donante); si ese donante
        no alcanza para el faltante, la emergencia se bloquea;
      - si no hay refuerzo o el donante no alcanza -> bloqueada.

    `zonas` es la lista ordenada de nombres de zona que fija el orden de las filas de
    `tasa_horaria_zona` (Z x 24) y de `unidades_muestra_zona` (lista de Z vectores).
    Las unidades prestadas se devuelven a su zona dueña al liberarse.

    Devuelve dict: p_bloqueo global (%), p_bloqueo_zona (dict), pct_refuerzo (%),
    atendidas, bloqueadas, refuerzos.
    """
    Z = len(zonas)
    flota = np.array([flota_por_zona[z] for z in zonas], dtype=np.int64)
    ocup = np.zeros(Z, dtype=np.int64)

    warmup_min = warmup * 24 * 60
    # Calendario: (tiempo, orden, tipo, dato). 'A'=llegada (dato=zona), 'L'=liberación (dato=zona dueña).
    heap = []
    orden = 0
    for zi in range(Z):
        for t in generar_llegadas(tasa_horaria_zona[zi], warmup + dias, rng):
            heapq.heappush(heap, (t, orden, 'A', zi)); orden += 1

    atendidas_z = np.zeros(Z, dtype=np.int64)
    bloqueadas_z = np.zeros(Z, dtype=np.int64)
    refuerzos = 0

    def agendar(owner, n, t):
        nonlocal orden
        for _ in range(n):
            dur = muestrear_servicio(media_serv, rng)
            heapq.heappush(heap, (t + dur, orden, 'L', owner)); orden += 1
        ocup[owner] += n

    while heap:
        t, _, tipo, dato = heapq.heappop(heap)
        if tipo == 'L':
            ocup[dato] -= 1
            if debug:
                assert 0 <= ocup[dato] <= flota[dato]
            continue

        zi = dato
        contar = t >= warmup_min
        need = muestrear_unidades(unidades_muestra_zona[zi], rng)
        libres_z = int(flota[zi] - ocup[zi])

        if libres_z >= need:                      # atención local
            agendar(zi, need, t)
            if contar:
                atendidas_z[zi] += 1
        elif refuerzo:                            # local + 1 donante
            faltan = need - libres_z
            donor, best = -1, -1
            for y in range(Z):
                if y == zi:
                    continue
                lib = int(flota[y] - ocup[y])
                if lib > best:
                    best, donor = lib, y
            if best >= faltan:
                if libres_z > 0:
                    agendar(zi, libres_z, t)
                agendar(donor, faltan, t)
                if contar:
                    atendidas_z[zi] += 1
                    refuerzos += 1
            elif contar:
                bloqueadas_z[zi] += 1
        elif contar:                              # sin refuerzo, sin cupo
            bloqueadas_z[zi] += 1

        if debug:
            assert 0 <= ocup[zi] <= flota[zi]

    atend_tot = int(atendidas_z.sum())
    bloq_tot = int(bloqueadas_z.sum())
    n_tot = atend_tot + bloq_tot
    p_bloqueo = 100.0 * bloq_tot / n_tot if n_tot else 0.0
    p_bloqueo_zona = {}
    for zi, z in enumerate(zonas):
        d = int(atendidas_z[zi] + bloqueadas_z[zi])
        p_bloqueo_zona[z] = 100.0 * bloqueadas_z[zi] / d if d else 0.0
    pct_refuerzo = 100.0 * refuerzos / atend_tot if atend_tot else 0.0
    return {'p_bloqueo': p_bloqueo, 'p_bloqueo_zona': p_bloqueo_zona,
            'pct_refuerzo': pct_refuerzo, 'atendidas': atend_tot,
            'bloqueadas': bloq_tot, 'refuerzos': refuerzos}

In [14]:
# Verificación rápida: el refuerzo debe reducir el bloqueo global
vol = df['zona'].value_counts()
TOTAL_DEMO = 14
flota_demo = {z: max(1, round(TOTAL_DEMO * vol[z] / vol.sum())) for z in ZONAS_ORDEN}
print('flota_demo:', flota_demo, '| total =', sum(flota_demo.values()))

for refu in (False, True):
    r = simular_replica_zonas(flota_demo, tasa_horaria_zona, unidades_muestra_zona,
                              media_serv=45, dias=30, warmup=3,
                              rng=np.random.default_rng(42), zonas=ZONAS_ORDEN,
                              refuerzo=refu, debug=True)
    print(f'refuerzo={refu!s:5s}  p_bloqueo_global={r["p_bloqueo"]:6.2f}%  '
          f'pct_refuerzo={r["pct_refuerzo"]:5.2f}%  (atendidas={r["atendidas"]}, bloqueadas={r["bloqueadas"]})')

flota_demo: {'Santiago': 5, 'Las Condes': 2, 'Providencia': 2, 'Estación Central': 1, 'Renca': 1, 'Independencia': 1, 'Vitacura': 1, 'Lo Barnechea': 1, 'Recoleta': 1, 'Resto RM': 1} | total = 16
refuerzo=False  p_bloqueo_global= 36.01%  pct_refuerzo= 0.00%  (atendidas=517, bloqueadas=291)
refuerzo=True   p_bloqueo_global=  2.72%  pct_refuerzo=36.51%  (atendidas=786, bloqueadas=22)


## 4. Verificación y validación

Evidencia de que el motor implementa correctamente el modelo:

- **5.1 Determinístico** — con flota muy grande, el bloqueo debe ser 0.
- **5.2 Saturación + monotonía** — con flota muy chica el bloqueo es alto; y más flota ⇒ menos bloqueo.
- **5.3 Conservación** — nunca `ocupadas > flota` (assert interno con `debug=True`).
- **5.4 Volumen** — el nº de llegadas simuladas en 365 días cae dentro de ±3% del real (10.082).
- **5.5 Perfil horario** — el perfil simulado reproduce el real (figura).

In [15]:
# 5.1 Test determinístico: flota enorme => p_bloqueo == 0
r_det = simular_replica_agregado(200, tasa_horaria, unidades_muestra, media_serv=45,
                                 dias=60, warmup=3, rng=np.random.default_rng(42))
print(f'flota=200 -> p_bloqueo = {r_det["p_bloqueo"]:.4f}%')
assert r_det['p_bloqueo'] == 0.0, 'Con flota=200 no debería haber bloqueo'
print('5.1 OK: bloqueo nulo con flota grande')

flota=200 -> p_bloqueo = 0.0000%


5.1 OK: bloqueo nulo con flota grande


In [16]:
# 5.2 Test de saturación (flota=2 => alto) y monotonía (más flota => menos bloqueo)
niveles = [2, 8, 20]
pb = []
for f in niveles:
    r = simular_replica_agregado(f, tasa_horaria, unidades_muestra, media_serv=45,
                                 dias=60, warmup=3, rng=np.random.default_rng(42))
    pb.append(r['p_bloqueo'])
    print(f'flota={f:2d} -> p_bloqueo = {r["p_bloqueo"]:.2f}%')
assert pb[0] > 30.0, f'flota=2 debería saturar (>30%), dio {pb[0]:.2f}%'
assert pb[0] > pb[1] > pb[2], f'p_bloqueo debería decrecer con la flota: {pb}'
print('5.2 OK: saturación y monotonía')

flota= 2 -> p_bloqueo = 49.61%


flota= 8 -> p_bloqueo = 3.63%
flota=20 -> p_bloqueo = 0.30%
5.2 OK: saturación y monotonía


In [17]:
# 5.3 Test de conservación: nunca ocupadas > flota (assert interno, debug=True)
simular_replica_agregado(6, tasa_horaria, unidades_muestra, media_serv=45,
                         dias=60, warmup=3, rng=np.random.default_rng(7), debug=True)
flota_chk = {z: max(1, v) for z, v in {**{k: 1 for k in ZONAS_ORDEN}, 'Santiago': 3}.items()}
simular_replica_zonas(flota_chk, tasa_horaria_zona, unidades_muestra_zona, media_serv=45,
                      dias=60, warmup=3, rng=np.random.default_rng(7), zonas=ZONAS_ORDEN,
                      refuerzo=True, debug=True)
print('5.3 OK: invariante 0 <= ocupadas <= flota respetada en ambos motores')

5.3 OK: invariante 0 <= ocupadas <= flota respetada en ambos motores

In [18]:
# 5.4 Validación de volumen: llegadas simuladas en 365 días vs real (10082)
VOL_REAL = 10082
n_sims = [len(generar_llegadas(tasa_horaria, 365, np.random.default_rng(1000 + s))) for s in range(10)]
vol_sim = float(np.mean(n_sims))
err = 100.0 * (vol_sim - VOL_REAL) / VOL_REAL
print(f'Volumen real: {VOL_REAL}')
print(f'Volumen simulado (media de 10 réplicas, 365 días): {vol_sim:.1f}  (rango {min(n_sims)}-{max(n_sims)})')
print(f'Error relativo: {err:+.2f}%')
assert abs(err) <= 3.0, f'El volumen simulado debería caer en ±3% (dio {err:+.2f}%)'
print('5.4 OK: volumen dentro de ±3%')

Volumen real: 10082
Volumen simulado (media de 10 réplicas, 365 días): 10106.9  (rango 9976-10315)
Error relativo: +0.25%
5.4 OK: volumen dentro de ±3%


In [19]:
# 5.5 Validación de perfil horario: simulado vs real (figura)
DIR_FIG = DIR_SALIDA / 'figuras'
DIR_FIG.mkdir(exist_ok=True)

perfil_real = df.groupby('hora').size().reindex(range(24), fill_value=0).to_numpy()
_ll = generar_llegadas(tasa_horaria, 365, np.random.default_rng(2025))
perfil_sim = np.bincount([int((t // 60) % 24) for t in _ll], minlength=24)

fig, ax = plt.subplots(figsize=(10, 4.5))
x = np.arange(24)
ax.bar(x - 0.2, perfil_real, width=0.4, label='Real (SIGEB 2025)', color='#3b6fb0')
ax.bar(x + 0.2, perfil_sim, width=0.4, label='Simulado (365 días)', color='#e0883b')
ax.set_xlabel('Hora del día')
ax.set_ylabel('Número de emergencias')
ax.set_title('Validación del perfil horario de llegadas: real vs simulado')
ax.set_xticks(x)
ax.legend()
fig.tight_layout()
RUTA_FIG = DIR_FIG / 'val_perfil_horario.png'
fig.savefig(RUTA_FIG, dpi=150)
plt.close(fig)
print('Figura guardada:', RUTA_FIG, '| existe:', RUTA_FIG.exists())

Figura guardada: C:\Users\josea\Downloads\uc\2026-1\MMEE\UC-2026-1_MMEE-T3\tarea-3\despacho_recursos\figuras\val_perfil_horario.png | existe: True
